# 18 — Geological CO₂ Storage Representation in GeoCANOE / Temoa

## Purpose

This notebook develops the **model representation of geological CO₂ storage** within the GeoCANOE / Temoa framework.

Notebook 17 established the spatial relationship between geological storage evidence and GeoCANOE model regions. The purpose of this notebook is to move from that **Silver-layer spatial evidence** toward the **Gold-layer optimization representation** required by Temoa.

The immediate questions are:

1. How should geological CO₂ injection be represented as a Temoa technology?
2. What commodities should connect captured and transported CO₂ to geological storage?
3. How should annual **injectivity** and cumulative **storage capacity** be represented?
4. Which existing GeoCANOE registry files must be extended?
5. Which storage attributes belong in Silver products versus Gold/schema construction?
6. How should geological storage interact with future emissions accounting, carbon-removal accounting, and net-zero constraints?

The initial working technology is:

`CO2_INJECT`

with the conceptual transformation:

`co2 -> CO2_INJECT -> co2_stored`

This notebook is intentionally exploratory. The objective is to establish the model semantics and registry/schema requirements before implementing the Silver-to-Gold storage encoding workflow.

---

## 1. Separate physical CO₂ flows from carbon accounting

A key design principle is to distinguish three related but different model concepts.

### 1.1 Physical CO₂ commodity flow

Captured CO₂ already exists within GeoCANOE as a model commodity that can be produced by point-source capture technologies and, later, direct-air-capture technologies.

Geological injection can therefore initially be represented as a conventional Temoa transformation:

`co2 -> CO2_INJECT -> co2_stored`

where:

- `co2` represents captured CO₂ available for transportation or utilization;
- `CO2_INJECT` represents the injection and geological-storage process;
- `co2_stored` represents CO₂ that has passed through the geological injection boundary.

This follows the fundamental Temoa representation of technologies as processes that transform input commodities into output commodities.

`co2_stored` should initially be interpreted as a **physical bookkeeping commodity**, rather than directly as an emissions credit or net-zero accounting variable.

---

### 1.2 Geological storage constraints

A geological reservoir has at least two fundamentally different physical constraints:

1. **Injectivity** — the maximum rate at which CO₂ can be injected during a model period.
2. **Storage capacity** — the cumulative amount of CO₂ that can ultimately be stored within the reservoir.

These should not be collapsed into a single parameter.

The NCAF strategic-planning formulation similarly separates these quantities. Annual reservoir injection is constrained by reservoir injectivity, while the cumulative quantity injected across the planning horizon is constrained by the reservoir's operational lifetime storage capacity.

For GeoCANOE, a preliminary Temoa mapping to investigate is:

| Geological concept | Candidate Temoa representation |
| --- | --- |
| Annual injection rate | `limit_activity` and/or installed `CO2_INJECT` capacity |
| Injection infrastructure capacity | `limit_capacity`, `existing_capacity`, or endogenous `CO2_INJECT` capacity |
| Cumulative geological storage capacity | `limit_resource` |
| Injection CAPEX | `cost_invest` |
| Fixed injection OPEX | `cost_fixed` |
| Variable injection/storage cost | `cost_variable` |
| Reservoir availability | regional technology availability during schema construction |

The exact interpretation of these tables must be verified against Temoa's activity and capacity conventions before implementation.

---

## 2. Initial technology representation

The simplest first-pass representation is:

```text
captured / transported CO2
          |
          v
    +-------------+
    | CO2_INJECT  |
    +-------------+
          |
          v
      co2_stored

POINT-SOURCE CARBON

gross point-source emissions
          |
          +--------------------------> atmosphere
          |
          +---- capture
                  |
                  v
             captured CO2
                  |
           +------+------+
           |             |
           v             v
       injection     fuel production
           |             |
           v             v
        storage       fuel demand
                          |
                          v
                      atmosphere

ATMOSPHERIC CARBON

atmosphere
    |
    v
   DAC
    |
    v
captured CO2
    |
 +--+-------------+
 |                |
 v                v
injection     fuel production
 |                |
 v                v
storage         fuel demand
                    |
                    v
                atmosphere

Before adding geological storage technologies or commodities, the current GeoCANOE registry inputs should be inspected directly.

The immediate objective is to identify the existing conventions used for:

- technology identifiers;
- commodity identifiers;
- capitalization;
- technology categories and sub-categories;
- sector labels;
- input and output commodities;
- efficiency definitions;
- transport technologies;
- CO₂-related technologies and commodities already present in the model.

This step is important because proposed names such as:

```text
CO2_INJECT
co2_stored

In [1]:
# ---------------------------------------------------------------------------
# Cell 2 — Inspect current registry conventions
# ---------------------------------------------------------------------------

from pathlib import Path

import sqlite3
import pyogrio
import pandas as pd
import geopandas as gpd



# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------

PROJECT_ROOT = Path.cwd()

# If the notebook is running from notebooks/ or another subdirectory,
# walk upward until the registry directory is found.
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "registry").is_dir():
        PROJECT_ROOT = candidate
        break

REGISTRY_DIR = PROJECT_ROOT / "registry"

registry_paths = {
    "techs": REGISTRY_DIR / "techs.csv",
    "commodities": REGISTRY_DIR / "commodities.csv",
    "generation_efficiency": REGISTRY_DIR / "generation_efficiency.csv",
    "transport_techs": REGISTRY_DIR / "transport_techs.csv",
}


# ---------------------------------------------------------------------------
# Load available registry tables
# ---------------------------------------------------------------------------

registry_tables = {}

for name, path in registry_paths.items():
    if path.is_file():
        registry_tables[name] = pd.read_csv(path)
        print(f"\nLoaded {name}: {path}")
        print(f"Shape: {registry_tables[name].shape}")
        print(f"Columns: {registry_tables[name].columns.tolist()}")
    else:
        print(f"\n[Missing] {name}: {path}")


# ---------------------------------------------------------------------------
# Inspect CO2-, storage-, DAC-, and fuel-related rows
# ---------------------------------------------------------------------------

search_terms = [
    "co2",
    "capture",
    "inject",
    "storage",
    "dac",
    "gasoline",
    "gsl",
    "fuel",
]

for name, df in registry_tables.items():
    text_view = df.astype(str).apply(
        lambda column: column.str.lower()
    )

    mask = pd.Series(False, index=df.index)

    for term in search_terms:
        mask |= text_view.apply(
            lambda column: column.str.contains(
                term,
                na=False,
                regex=False,
            )
        ).any(axis=1)

    relevant = df.loc[mask].copy()

    print("\n" + "=" * 80)
    print(name.upper())
    print("=" * 80)

    if relevant.empty:
        print("No matching rows found.")
    else:
        display(relevant)


Loaded techs: c:\Users\aviga\Research\repos\temoa-upstream\temoa_geospace\registry\techs.csv
Shape: (16, 5)
Columns: ['tech', 'flag', 'annual', 'exchange', 'unlim_cap']

Loaded commodities: c:\Users\aviga\Research\repos\temoa-upstream\temoa_geospace\registry\commodities.csv
Shape: (7, 3)
Columns: ['name', 'flag', 'description']

Loaded generation_efficiency: c:\Users\aviga\Research\repos\temoa-upstream\temoa_geospace\registry\generation_efficiency.csv
Shape: (9, 4)
Columns: ['tech', 'input_comm', 'output_comm', 'efficiency']

Loaded transport_techs: c:\Users\aviga\Research\repos\temoa-upstream\temoa_geospace\registry\transport_techs.csv
Shape: (9, 5)
Columns: ['tech', 'input_comm', 'output_comm', 'cost_per_km', 'intercept_cost_per_km']

TECHS


,tech,flag,annual,exchange,unlim_cap
2,CO2_CAP,p,1,0,0
4,GSL_PLANT,p,1,0,0
7,CO2_PIPE,p,1,1,0
9,GSL_PIPE,p,1,1,0
10,GSL_DEMAND,p,1,0,1
11,GSL_BACKUP,p,1,0,1
12,CO2_TRUCK,p,1,1,0
15,GSL_TRUCK,p,1,1,0



COMMODITIES


,name,flag,description
2,co2,wa,co2 captured
3,gsl,wa,gasoline
5,d_gsl,d,gasoline demand



GENERATION_EFFICIENCY


,tech,input_comm,output_comm,efficiency
2,CO2_CAP,ethos,co2,1.000000
3,METOH_PLANT,co2,ch3oh,0.714286
6,GSL_PLANT,ch3oh,gsl,0.444444
7,GSL_PLANT,h2,gsl,200.000000
8,GSL_BACKUP,ethos,d_gsl,1.000000



TRANSPORT_TECHS


,tech,input_comm,output_comm,cost_per_km,intercept_cost_per_km
2,CO2_PIPE,co2,co2,0.0196,0.0
3,GSL_PIPE,gsl,gsl,0.0196,0.0
5,CO2_TRUCK,co2,co2,0.0100,0.0
8,GSL_TRUCK,gsl,gsl,0.0100,0.0


In [2]:
# ---------------------------------------------------------------------------
# Cell 4 — Trace current commodity and technology relationships
# ---------------------------------------------------------------------------

techs = registry_tables["techs"]
commodities = registry_tables["commodities"]
generation_efficiency = registry_tables["generation_efficiency"]
transport_techs = registry_tables["transport_techs"]


# ---------------------------------------------------------------------------
# Inspect all commodities
# ---------------------------------------------------------------------------

print("=" * 80)
print("COMMODITIES")
print("=" * 80)

display(commodities)


# ---------------------------------------------------------------------------
# Inspect all technologies
# ---------------------------------------------------------------------------

print("\n" + "=" * 80)
print("TECHNOLOGIES")
print("=" * 80)

display(techs)


# ---------------------------------------------------------------------------
# Trace all generation/process relationships involving ethos
# ---------------------------------------------------------------------------

ethos_rows = generation_efficiency[
    (generation_efficiency["input_comm"] == "ethos")
    | (generation_efficiency["output_comm"] == "ethos")
]

print("\n" + "=" * 80)
print("ETHOS-RELATED PROCESS RELATIONSHIPS")
print("=" * 80)

display(ethos_rows)


# ---------------------------------------------------------------------------
# Trace the current carbon and gasoline chains
# ---------------------------------------------------------------------------

tracked_commodities = {
    "ethos",
    "co2",
    "ch3oh",
    "gsl",
    "d_gsl",
}

tracked_generation = generation_efficiency[
    generation_efficiency["input_comm"].isin(tracked_commodities)
    | generation_efficiency["output_comm"].isin(tracked_commodities)
].copy()

tracked_transport = transport_techs[
    transport_techs["input_comm"].isin(tracked_commodities)
    | transport_techs["output_comm"].isin(tracked_commodities)
].copy()

print("\n" + "=" * 80)
print("CURRENT CARBON / FUEL PROCESS CHAIN")
print("=" * 80)

display(tracked_generation)

print("\n" + "=" * 80)
print("CURRENT CARBON / FUEL TRANSPORT CHAIN")
print("=" * 80)

display(tracked_transport)

COMMODITIES


,name,flag,description
0,h2,wa,hydrogen
1,ch3oh,wa,methanol
2,co2,wa,co2 captured
3,gsl,wa,gasoline
4,elc,wa,electricity
5,d_gsl,d,gasoline demand
6,ethos,s,dummy



TECHNOLOGIES


,tech,flag,annual,exchange,unlim_cap
0,ELC_GEN,p,1,0,0
1,H2_PLANT,p,1,0,0
2,CO2_CAP,p,1,0,0
3,METOH_PLANT,p,1,0,0
4,GSL_PLANT,p,1,0,0
5,ELC_TRANS,p,1,1,0
6,H2_PIPE,p,1,1,0
7,CO2_PIPE,p,1,1,0
8,METOH_PIPE,p,1,1,0
9,GSL_PIPE,p,1,1,0



ETHOS-RELATED PROCESS RELATIONSHIPS


,tech,input_comm,output_comm,efficiency
0,ELC_GEN,ethos,elc,1.0
2,CO2_CAP,ethos,co2,1.0
8,GSL_BACKUP,ethos,d_gsl,1.0



CURRENT CARBON / FUEL PROCESS CHAIN


,tech,input_comm,output_comm,efficiency
0,ELC_GEN,ethos,elc,1.000000
2,CO2_CAP,ethos,co2,1.000000
3,METOH_PLANT,co2,ch3oh,0.714286
4,METOH_PLANT,h2,ch3oh,5.208333
5,METOH_PLANT,elc,ch3oh,5.714286
6,GSL_PLANT,ch3oh,gsl,0.444444
7,GSL_PLANT,h2,gsl,200.000000
8,GSL_BACKUP,ethos,d_gsl,1.000000



CURRENT CARBON / FUEL TRANSPORT CHAIN


,tech,input_comm,output_comm,cost_per_km,intercept_cost_per_km
1,METOH_PIPE,ch3oh,ch3oh,0.0196,0.0
2,CO2_PIPE,co2,co2,0.0196,0.0
3,GSL_PIPE,gsl,gsl,0.0196,0.0
5,CO2_TRUCK,co2,co2,0.0100,0.0
7,METOH_TRUCK,ch3oh,ch3oh,0.0100,0.0
8,GSL_TRUCK,gsl,gsl,0.0100,0.0


## 8. Existing gasoline fallback representation

The current registry includes:

```text
GSL_BACKUP

In [3]:
# ---------------------------------------------------------------------------
# Cell 5 — Draft candidate CO2 injection registry additions
# ---------------------------------------------------------------------------


# ---------------------------------------------------------------------------
# Candidate technology row
# ---------------------------------------------------------------------------

candidate_tech = pd.DataFrame(
    [
        {
            "tech": "CO2_INJECT",
            "flag": "p",
            "annual": 1,
            "exchange": 0,
            "unlim_cap": 0,
        }
    ]
)


# ---------------------------------------------------------------------------
# Candidate stored-CO2 commodity row
# ---------------------------------------------------------------------------

candidate_commodity = pd.DataFrame(
    [
        {
            "name": "co2_stored",
            "flag": "wa",  # provisional until sink/terminal semantics are confirmed
            "description": "geologically stored co2",
        }
    ]
)


# ---------------------------------------------------------------------------
# Candidate process-efficiency row
# ---------------------------------------------------------------------------

candidate_efficiency = pd.DataFrame(
    [
        {
            "tech": "CO2_INJECT",
            "input_comm": "co2",
            "output_comm": "co2_stored",
            "efficiency": 1.0,
        }
    ]
)


# ---------------------------------------------------------------------------
# Display candidate additions beside current registry schemas
# ---------------------------------------------------------------------------

print("=" * 80)
print("CANDIDATE TECHS.CSV ADDITION")
print("=" * 80)
display(candidate_tech)

print("\n" + "=" * 80)
print("CANDIDATE COMMODITIES.CSV ADDITION")
print("=" * 80)
display(candidate_commodity)

print("\n" + "=" * 80)
print("CANDIDATE GENERATION_EFFICIENCY.CSV ADDITION")
print("=" * 80)
display(candidate_efficiency)


# ---------------------------------------------------------------------------
# Basic schema checks
# ---------------------------------------------------------------------------

checks = {
    "tech columns match": list(candidate_tech.columns)
    == list(registry_tables["techs"].columns),

    "commodity columns match": list(candidate_commodity.columns)
    == list(registry_tables["commodities"].columns),

    "efficiency columns match": list(candidate_efficiency.columns)
    == list(registry_tables["generation_efficiency"].columns),

    "CO2_INJECT not already present": "CO2_INJECT"
    not in set(registry_tables["techs"]["tech"]),

    "co2_stored not already present": "co2_stored"
    not in set(registry_tables["commodities"]["name"]),
}

print("\n" + "=" * 80)
print("SCHEMA CHECKS")
print("=" * 80)

for check, passed in checks.items():
    print(f"{check}: {passed}")

CANDIDATE TECHS.CSV ADDITION


,tech,flag,annual,exchange,unlim_cap
0,CO2_INJECT,p,1,0,0



CANDIDATE COMMODITIES.CSV ADDITION


,name,flag,description
0,co2_stored,wa,geologically stored co2



CANDIDATE GENERATION_EFFICIENCY.CSV ADDITION


,tech,input_comm,output_comm,efficiency
0,CO2_INJECT,co2,co2_stored,1.0



SCHEMA CHECKS
tech columns match: True
commodity columns match: True
efficiency columns match: True
CO2_INJECT not already present: True
co2_stored not already present: True


### Stored CO₂ commodity flag

The current Temoa v3 commodity flag:

```text
wa

In [4]:
# ---------------------------------------------------------------------------
# Cell 6 — Inspect current storage-relevant Temoa constraint tables
# ---------------------------------------------------------------------------


# ---------------------------------------------------------------------------
# Locate the baseline / schema SQLite database
# ---------------------------------------------------------------------------

candidate_db_paths = [
    PROJECT_ROOT / "data_files" / "CANOE_geospatial.sqlite",
    PROJECT_ROOT / "data_files" / "canoe_dataset_schema.sqlite",
    PROJECT_ROOT / "data_files" / "processed" / "schema" / "CANOE_geospatial.sqlite",
]

db_path = next(
    (path for path in candidate_db_paths if path.is_file()),
    None,
)

if db_path is None:
    print("No candidate SQLite database found automatically.")
else:
    print(f"Using database: {db_path}")


# ---------------------------------------------------------------------------
# Inspect storage-relevant table schemas
# ---------------------------------------------------------------------------

tables_to_inspect = [
    "LimitActivity",
    "LimitCapacity",
    "LimitResource",
    "CostInvest",
    "CostFixed",
    "CostVariable",
]

if db_path is not None:
    with sqlite3.connect(db_path) as conn:

        available_tables = pd.read_sql_query(
            """
            SELECT name
            FROM sqlite_master
            WHERE type = 'table'
            ORDER BY name
            """,
            conn,
        )["name"].tolist()

        print("\nAvailable target tables:")
        for table in tables_to_inspect:
            print(f"  {table}: {table in available_tables}")

        for table in tables_to_inspect:
            if table not in available_tables:
                continue

            print("\n" + "=" * 80)
            print(table.upper())
            print("=" * 80)

            schema = pd.read_sql_query(
                f"PRAGMA table_info({table})",
                conn,
            )

            display(schema)

            rows = pd.read_sql_query(
                f"SELECT * FROM {table} LIMIT 10",
                conn,
            )

            if rows.empty:
                print("No rows currently populated.")
            else:
                display(rows)

Using database: c:\Users\aviga\Research\repos\temoa-upstream\temoa_geospace\data_files\CANOE_geospatial.sqlite

Available target tables:
  LimitActivity: True
  LimitCapacity: True
  LimitResource: True
  CostInvest: True
  CostFixed: True
  CostVariable: True

LIMITACTIVITY


,cid,name,type,notnull,dflt_value,pk
0,0,region,TEXT,0,None,1
1,1,period,INTEGER,0,None,2
2,2,tech_or_group,TEXT,0,None,3
3,3,operator,TEXT,1,"""le""",4
4,4,activity,REAL,0,None,0
5,5,units,TEXT,0,None,0
6,6,notes,TEXT,0,None,0
7,7,data_source,TEXT,0,None,0
8,8,dq_cred,INTEGER,0,None,0
9,9,dq_geog,INTEGER,0,None,0


No rows currently populated.

LIMITCAPACITY


,cid,name,type,notnull,dflt_value,pk
0,0,region,TEXT,0,None,1
1,1,period,INTEGER,0,None,2
2,2,tech_or_group,TEXT,0,None,3
3,3,operator,TEXT,1,"""le""",4
4,4,capacity,REAL,0,None,0
5,5,units,TEXT,0,None,0
6,6,notes,TEXT,0,None,0
7,7,data_source,TEXT,0,None,0
8,8,dq_cred,INTEGER,0,None,0
9,9,dq_geog,INTEGER,0,None,0


No rows currently populated.

LIMITRESOURCE


...

No rows currently populated.

COSTFIXED


,cid,name,type,notnull,dflt_value,pk
0,0,region,TEXT,1,None,1
1,1,period,INTEGER,1,None,2
2,2,tech,TEXT,1,None,3
3,3,vintage,INTEGER,1,None,4
4,4,cost,REAL,0,None,0
5,5,units,TEXT,0,None,0
6,6,notes,TEXT,0,None,0
7,7,data_source,TEXT,0,None,0
8,8,dq_cred,INTEGER,0,None,0
9,9,dq_geog,INTEGER,0,None,0


No rows currently populated.

COSTVARIABLE


,cid,name,type,notnull,dflt_value,pk
0,0,region,TEXT,1,None,1
1,1,period,INTEGER,1,None,2
2,2,tech,TEXT,1,None,3
3,3,vintage,INTEGER,1,None,4
4,4,cost,REAL,0,None,0
5,5,units,TEXT,0,None,0
6,6,notes,TEXT,0,None,0
7,7,data_source,TEXT,0,None,0
8,8,dq_cred,INTEGER,0,None,0
9,9,dq_geog,INTEGER,0,None,0


No rows currently populated.


### 10.6 `LimitResource` and foresight mode

Temoa defines `LimitResource` as a:

> cumulative activity limit across time periods

with `tech_or_group` accepting either a technology name or technology-group name.

This makes `LimitResource` directly suitable for representing geological CO₂ storage capacity:

```text
cumulative CO2_INJECT activity
    <= geological storage capacity

## Gold-schema integration point

Inspection of the current GeoCANOE schema builder shows that geological storage should be added as a new node-level Gold encoding step rather than folded into the existing transport or point-source logic.

The current schema workflow already rebuilds several node-level model tables from spatially assigned inputs.

For example, point-source CO₂ and electricity availability are currently encoded through:

```text
rebuild_capacity_limits()
    |
    +--> LimitCapacity: CO2_CAP
    |
    +--> LimitCapacity: ELC_GEN

In [5]:
# ---------------------------------------------------------------------------
# Cell 7 — Load provinces_only 25 km CO2-storage Silver test case
# ---------------------------------------------------------------------------

PROCESSED_STORAGE_DIR = (
    PROJECT_ROOT
    / "data_files"
    / "processed"
    / "co2_storage"
)


# ---------------------------------------------------------------------------
# Find the provinces_only projected 25 km storage product
# ---------------------------------------------------------------------------

matches = sorted(
    PROCESSED_STORAGE_DIR.glob(
        "*provinces_only*25km*_co2_storage.gpkg"
    )
)

print("=" * 80)
print("PROVINCES_ONLY 25 KM STORAGE PRODUCT")
print("=" * 80)

if len(matches) == 0:
    raise FileNotFoundError(
        "No provinces_only 25 km CO2-storage GeoPackage was found in "
        f"{PROCESSED_STORAGE_DIR}"
    )

if len(matches) > 1:
    print("Multiple candidate files found:")
    for path in matches:
        print(f"  {path.name}")

    raise ValueError(
        "Expected exactly one provinces_only 25 km storage product."
    )

storage_gpkg = matches[0]

print(f"Selected: {storage_gpkg}")
print()


# ---------------------------------------------------------------------------
# Confirm expected Silver layers
# ---------------------------------------------------------------------------

layers = pd.DataFrame(
    pyogrio.list_layers(storage_gpkg),
    columns=["layer", "geometry_type"],
)

display(layers)

required_layers = {
    "regional_storage_evidence",
    "storage_region_crosswalk",
}

available_layers = set(layers["layer"])

missing_layers = required_layers - available_layers

if missing_layers:
    raise ValueError(
        f"Storage GeoPackage is missing expected layers: "
        f"{sorted(missing_layers)}"
    )


# ---------------------------------------------------------------------------
# Load Silver storage products
# ---------------------------------------------------------------------------

storage_regions = gpd.read_file(
    storage_gpkg,
    layer="regional_storage_evidence",
)

storage_crosswalk = gpd.read_file(
    storage_gpkg,
    layer="storage_region_crosswalk",
)


# ---------------------------------------------------------------------------
# Basic test-case summary
# ---------------------------------------------------------------------------

print("\n" + "=" * 80)
print("REGIONAL STORAGE EVIDENCE")
print("=" * 80)

print(f"Rows: {len(storage_regions):,}")
print(f"CRS:  {storage_regions.crs}")
print(f"Columns: {len(storage_regions.columns)}")

display(storage_regions.head())


print("\n" + "=" * 80)
print("STORAGE REGION CROSSWALK")
print("=" * 80)

print(f"Rows: {len(storage_crosswalk):,}")
print(f"CRS:  {storage_crosswalk.crs}")
print(f"Columns: {len(storage_crosswalk.columns)}")

display(storage_crosswalk.head())


# ---------------------------------------------------------------------------
# Storage-accessibility summary
# ---------------------------------------------------------------------------

print("\n" + "=" * 80)
print("STORAGE ACCESSIBILITY SUMMARY")
print("=" * 80)

if "storage_accessible" in storage_regions.columns:
    accessibility_summary = (
        storage_regions["storage_accessible"]
        .value_counts(dropna=False)
        .rename_axis("storage_accessible")
        .reset_index(name="regions")
    )

    display(accessibility_summary)


# ---------------------------------------------------------------------------
# Quantitative evidence summary
# ---------------------------------------------------------------------------

evidence_columns = [
    "has_any_capacity_evidence",
    "has_quantitative_storage_evidence",
    "capacity_evidence_coverage_fraction",
]

available_evidence_columns = [
    column
    for column in evidence_columns
    if column in storage_regions.columns
]

if available_evidence_columns:
    print("\n" + "=" * 80)
    print("QUANTITATIVE STORAGE EVIDENCE")
    print("=" * 80)

    display(
        storage_regions[
            ["region", *available_evidence_columns]
        ].head(20)
    )


# ---------------------------------------------------------------------------
# Crosswalk fields relevant to future Gold encoding
# ---------------------------------------------------------------------------

gold_candidate_fields = [
    "region",
    "site_id",
    "storage_feature_id",
    "storage_unit_id",
    "source_dataset",
    "storage_type",
    "representation",
    "assessment_type",
    "data_class",
    "capacity_data",
    "injectivity_status",
    "region_overlap_fraction",
    "feature_overlap_fraction",
]

available_gold_fields = [
    column
    for column in gold_candidate_fields
    if column in storage_crosswalk.columns
]

print("\n" + "=" * 80)
print("CROSSWALK FIELDS RELEVANT TO GOLD ENCODING")
print("=" * 80)

display(
    storage_crosswalk[
        available_gold_fields
    ].head(20)
)

PROVINCES_ONLY 25 KM STORAGE PRODUCT
Selected: c:\Users\aviga\Research\repos\temoa-upstream\temoa_geospace\data_files\processed\co2_storage\provinces_only_basemap_25km_centroid_co2_storage.gpkg



,layer,geometry_type
0,regional_storage_evidence,Polygon
1,storage_region_crosswalk,MultiPolygon



REGIONAL STORAGE EVIDENCE
Rows: 9,269
CRS:  EPSG:3347
Columns: 22


,region,site_id,atlantic_coverage_fraction,bc_coverage_fraction,natcarb_coverage_fraction,has_natcarb,has_bc_storage_atlas,has_atlantic_cos,has_quantitative_storage_evidence,has_qualitative_storage_evidence,...,has_p10_capacity,has_p50_capacity,has_p90_capacity,has_theoretical_capacity,has_effective_capacity,has_any_capacity_evidence,capacity_evidence_area_m2,capacity_evidence_coverage_fraction,capacity_evidence_coverage_percent,geometry
0,R0,R0,0.0,0.0,0.095111,True,False,False,True,False,...,True,True,True,False,False,True,3.699152e+06,0.005919,0.591864,"POLYGON ((6975000 700000, 6975000 725000, 6950..."
1,R1,R1,0.0,0.0,0.000000,False,False,False,False,False,...,False,False,False,False,False,False,0.000000e+00,0.000000,0.000000,"POLYGON ((7000000 700000, 7000000 725000, 6975..."
2,R2,R2,0.0,0.0,0.000000,False,False,False,False,False,...,False,False,False,False,False,False,0.000000e+00,0.000000,0.000000,"POLYGON ((7025000 725000, 7025000 750000, 7000..."
3,R3,R3,0.0,0.0,0.000000,False,False,False,False,False,...,False,False,False,False,False,False,0.000000e+00,0.000000,0.000000,"POLYGON ((7050000 725000, 7050000 750000, 7025..."
4,R4,R4,0.0,0.0,0.556138,True,False,False,True,False,...,True,True,True,False,False,True,6.533345e+07,0.104534,10.453352,"POLYGON ((7000000 750000, 7000000 775000, 6975..."



STORAGE REGION CROSSWALK
Rows: 61,748
CRS:  EPSG:3347
Columns: 18


...

,storage_accessible,regions
0,False,6534
1,True,2735



QUANTITATIVE STORAGE EVIDENCE


,region,has_any_capacity_evidence,has_quantitative_storage_evidence,capacity_evidence_coverage_fraction
0,R0,True,True,0.005919
1,R1,False,False,0.000000
2,R2,False,False,0.000000
3,R3,False,False,0.000000
4,R4,True,True,0.104534
5,R5,False,False,0.000000
6,R6,False,False,0.000000
7,R7,False,False,0.000000
8,R8,False,False,0.000000
9,R9,False,False,0.000000



CROSSWALK FIELDS RELEVANT TO GOLD ENCODING


,region,site_id,storage_feature_id,storage_unit_id,source_dataset,storage_type,representation,assessment_type,data_class,capacity_data,injectivity_status,region_overlap_fraction,feature_overlap_fraction
0,R0,R0,NATCARB_SALINE_natcarb:saline_cell:941199,NAT_SAL_311f7d63fa70,NATCARB,saline_aquifer,resource_grid_cell,quantitative_geological_storage_resource,geological_storage_capacity,1.0,not_quantitatively_assessed,0.001676,0.009839
1,R0,R0,NATCARB_SALINE_natcarb:saline_cell:941200,NAT_SAL_311f7d63fa70,NATCARB,saline_aquifer,resource_grid_cell,quantitative_geological_storage_resource,geological_storage_capacity,1.0,not_quantitatively_assessed,0.004242,0.024925
2,R0,R0,NATCARB_SALINE_natcarb:saline_cell:941255,NAT_SAL_311f7d63fa70,NATCARB,saline_aquifer,resource_grid_cell,quantitative_geological_storage_resource,geological_storage_capacity,1.0,not_quantitatively_assessed,0.089193,0.523917
3,R0,R0,NATCARB_SALINE_natcarb:saline_cell:936129,NAT_SAL_5361b04c6542,NATCARB,saline_aquifer,resource_grid_cell,quantitative_geological_storage_resource,geological_storage_capacity,1.0,not_quantitatively_assessed,0.001676,0.009839
4,R0,R0,NATCARB_SALINE_natcarb:saline_cell:936130,NAT_SAL_5361b04c6542,NATCARB,saline_aquifer,resource_grid_cell,quantitative_geological_storage_resource,geological_storage_capacity,1.0,not_quantitatively_assessed,0.004242,0.024925
5,R0,R0,NATCARB_SALINE_natcarb:saline_cell:936149,NAT_SAL_5361b04c6542,NATCARB,saline_aquifer,resource_grid_cell,quantitative_geological_storage_resource,geological_storage_capacity,1.0,not_quantitatively_assessed,0.089193,0.523917
6,R0,R0,NATCARB_SALINE_natcarb:saline_cell:947581,NAT_SAL_7d4667380b63,NATCARB,saline_aquifer,resource_grid_cell,quantitative_geological_storage_resource,geological_storage_capacity,1.0,not_quantitatively_assessed,0.001676,0.009839
7,R0,R0,NATCARB_SALINE_natcarb:saline_cell:947582,NAT_SAL_7d4667380b63,NATCARB,saline_aquifer,resource_grid_cell,quantitative_geological_storage_resource,geological_storage_capacity,1.0,not_quantitatively_assessed,0.004242,0.024925
8,R0,R0,NATCARB_SALINE_natcarb:saline_cell:947584,NAT_SAL_7d4667380b63,NATCARB,saline_aquifer,resource_grid_cell,quantitative_geological_storage_resource,geological_storage_capacity,1.0,not_quantitatively_assessed,0.089193,0.523917
9,R0,R0,NATCARB_SALINE_natcarb:saline_cell:946065,NAT_SAL_8dbf6c8c8487,NATCARB,saline_aquifer,resource_grid_cell,quantitative_geological_storage_resource,geological_storage_capacity,1.0,not_quantitatively_assessed,0.001676,0.009839


In [6]:
# ---------------------------------------------------------------------------
# Cell 8 — Classify Silver storage evidence for Gold readiness
# ---------------------------------------------------------------------------


# ---------------------------------------------------------------------------
# Region-level eligibility for CO2_INJECT
# ---------------------------------------------------------------------------

gold_storage_regions = storage_regions[
    [
        "region",
        "site_id",
        "storage_accessible",
        "has_quantitative_storage_evidence",
        "has_qualitative_storage_evidence",
        "has_any_capacity_evidence",
        "capacity_evidence_coverage_fraction",
    ]
].copy()


gold_storage_regions["co2_inject_available"] = (
    gold_storage_regions["storage_accessible"]
)


# ---------------------------------------------------------------------------
# Determine whether quantitative Gold constraints are currently supported
# ---------------------------------------------------------------------------

gold_storage_regions["resource_limit_ready"] = False
gold_storage_regions["activity_limit_ready"] = False


# ---------------------------------------------------------------------------
# Add explanatory status
# ---------------------------------------------------------------------------

def classify_storage_status(row):
    if not row["storage_accessible"]:
        return "no_storage_evidence"

    if row["has_any_capacity_evidence"]:
        return "storage_available_capacity_evidence_unallocated"

    if row["has_qualitative_storage_evidence"]:
        return "storage_available_qualitative_only"

    return "storage_available_other_evidence"


gold_storage_regions["gold_storage_status"] = (
    gold_storage_regions.apply(
        classify_storage_status,
        axis=1,
    )
)


# ---------------------------------------------------------------------------
# Summarize Gold readiness
# ---------------------------------------------------------------------------

print("=" * 80)
print("GOLD STORAGE READINESS — PROVINCES_ONLY 25 KM")
print("=" * 80)

summary = (
    gold_storage_regions["gold_storage_status"]
    .value_counts()
    .rename_axis("gold_storage_status")
    .reset_index(name="regions")
)

summary["share_percent"] = (
    summary["regions"]
    / len(gold_storage_regions)
    * 100
)

display(summary)


print("\n" + "=" * 80)
print("CO2_INJECT AVAILABILITY")
print("=" * 80)

print(
    f"Regions eligible for CO2_INJECT: "
    f"{gold_storage_regions['co2_inject_available'].sum():,}"
)

print(
    f"Total model regions: "
    f"{len(gold_storage_regions):,}"
)


print("\n" + "=" * 80)
print("CURRENT QUANTITATIVE CONSTRAINT READINESS")
print("=" * 80)

print(
    "LimitResource ready:",
    gold_storage_regions["resource_limit_ready"].any(),
)

print(
    "LimitActivity ready:",
    gold_storage_regions["activity_limit_ready"].any(),
)


# ---------------------------------------------------------------------------
# Preview candidate Gold availability table
# ---------------------------------------------------------------------------

display(
    gold_storage_regions[
        [
            "region",
            "co2_inject_available",
            "gold_storage_status",
            "has_quantitative_storage_evidence",
            "has_qualitative_storage_evidence",
            "capacity_evidence_coverage_fraction",
        ]
    ].head(30)
)

GOLD STORAGE READINESS — PROVINCES_ONLY 25 KM


,gold_storage_status,regions,share_percent
0,no_storage_evidence,6534,70.493041
1,storage_available_capacity_evidence_unallocated,1483,15.999568
2,storage_available_other_evidence,1096,11.824361
3,storage_available_qualitative_only,156,1.683029



CO2_INJECT AVAILABILITY
Regions eligible for CO2_INJECT: 2,735
Total model regions: 9,269

CURRENT QUANTITATIVE CONSTRAINT READINESS
LimitResource ready: False
LimitActivity ready: False


,region,co2_inject_available,gold_storage_status,has_quantitative_storage_evidence,has_qualitative_storage_evidence,capacity_evidence_coverage_fraction
0,R0,True,storage_available_capacity_evidence_unallocated,True,False,0.005919
1,R1,False,no_storage_evidence,False,False,0.000000
2,R2,False,no_storage_evidence,False,False,0.000000
3,R3,False,no_storage_evidence,False,False,0.000000
4,R4,True,storage_available_capacity_evidence_unallocated,True,False,0.104534
5,R5,False,no_storage_evidence,False,False,0.000000
6,R6,False,no_storage_evidence,False,False,0.000000
7,R7,False,no_storage_evidence,False,False,0.000000
8,R8,False,no_storage_evidence,False,False,0.000000
9,R9,False,no_storage_evidence,False,False,0.000000


In [7]:
# ---------------------------------------------------------------------------
# Cell 9 — Build candidate CO2_INJECT regional availability
# ---------------------------------------------------------------------------

# Keep only regions where the Silver layer indicates storage accessibility.
co2_inject_regions = (
    gold_storage_regions.loc[
        gold_storage_regions["co2_inject_available"],
        [
            "region",
            "site_id",
            "gold_storage_status",
            "has_quantitative_storage_evidence",
            "has_qualitative_storage_evidence",
            "has_any_capacity_evidence",
            "capacity_evidence_coverage_fraction",
        ],
    ]
    .copy()
    .sort_values("region")
    .reset_index(drop=True)
)

print("=" * 80)
print("CANDIDATE CO2_INJECT REGIONS")
print("=" * 80)

print(f"Eligible regions: {len(co2_inject_regions):,}")
print(
    f"Share of all regions: "
    f"{len(co2_inject_regions) / len(gold_storage_regions) * 100:.2f}%"
)

display(co2_inject_regions.head(30))


# ---------------------------------------------------------------------------
# Breakdown by evidence class
# ---------------------------------------------------------------------------

print("\n" + "=" * 80)
print("CO2_INJECT REGIONS BY STORAGE EVIDENCE CLASS")
print("=" * 80)

inject_status_summary = (
    co2_inject_regions["gold_storage_status"]
    .value_counts()
    .rename_axis("gold_storage_status")
    .reset_index(name="regions")
)

inject_status_summary["share_of_inject_regions_percent"] = (
    inject_status_summary["regions"]
    / len(co2_inject_regions)
    * 100
)

display(inject_status_summary)


# ---------------------------------------------------------------------------
# Explicitly identify what is and is not ready for Gold encoding
# ---------------------------------------------------------------------------

gold_readiness = pd.DataFrame(
    [
        {
            "gold_component": "CO2_INJECT availability",
            "ready": True,
            "source": "storage_accessible",
            "interpretation": (
                "Region may host geological CO2 injection."
            ),
        },
        {
            "gold_component": "LimitResource",
            "ready": False,
            "source": "storage capacity assessments",
            "interpretation": (
                "Capacity evidence exists for some regions, but geological "
                "capacity has not yet been defensibly allocated to model regions."
            ),
        },
        {
            "gold_component": "LimitActivity",
            "ready": False,
            "source": "injectivity assessments",
            "interpretation": (
                "Current Silver evidence does not contain quantitative "
                "region-level injectivity values."
            ),
        },
        {
            "gold_component": "CostInvest / CostFixed / CostVariable",
            "ready": False,
            "source": "future geological storage TEA workflow",
            "interpretation": (
                "Injection cost workflow still to be developed."
            ),
        },
    ]
)

print("\n" + "=" * 80)
print("GOLD STORAGE COMPONENT READINESS")
print("=" * 80)

display(gold_readiness)

CANDIDATE CO2_INJECT REGIONS
Eligible regions: 2,735
Share of all regions: 29.51%


,region,site_id,gold_storage_status,has_quantitative_storage_evidence,has_qualitative_storage_evidence,has_any_capacity_evidence,capacity_evidence_coverage_fraction
0,R0,R0,storage_available_capacity_evidence_unallocated,True,False,True,0.005919
1,R1001,R1001,storage_available_qualitative_only,False,True,False,0.000000
2,R1002,R1002,storage_available_qualitative_only,False,True,False,0.000000
3,R1003,R1003,storage_available_qualitative_only,False,True,False,0.000000
4,R1004,R1004,storage_available_qualitative_only,False,True,False,0.000000
5,R1005,R1005,storage_available_qualitative_only,False,True,False,0.000000
6,R1006,R1006,storage_available_qualitative_only,False,True,False,0.000000
7,R1007,R1007,storage_available_qualitative_only,False,True,False,0.000000
8,R1008,R1008,storage_available_qualitative_only,False,True,False,0.000000
9,R1009,R1009,storage_available_qualitative_only,False,True,False,0.000000



CO2_INJECT REGIONS BY STORAGE EVIDENCE CLASS


,gold_storage_status,regions,share_of_inject_regions_percent
0,storage_available_capacity_evidence_unallocated,1483,54.223035
1,storage_available_other_evidence,1096,40.073126
2,storage_available_qualitative_only,156,5.703839



GOLD STORAGE COMPONENT READINESS


,gold_component,ready,source,interpretation
0,CO2_INJECT availability,True,storage_accessible,Region may host geological CO2 injection.
1,LimitResource,False,storage capacity assessments,"Capacity evidence exists for some regions, but..."
2,LimitActivity,False,injectivity assessments,Current Silver evidence does not contain quant...
3,CostInvest / CostFixed / CostVariable,False,future geological storage TEA workflow,Injection cost workflow still to be developed.


In [8]:
# ---------------------------------------------------------------------------
# Cell 10 — Diagnose storage evidence classes behind CO2_INJECT availability
# ---------------------------------------------------------------------------


# ---------------------------------------------------------------------------
# Join regional Gold status back onto the detailed Silver crosswalk
# ---------------------------------------------------------------------------

crosswalk_gold = storage_crosswalk.merge(
    gold_storage_regions[
        [
            "region",
            "gold_storage_status",
            "co2_inject_available",
        ]
    ],
    on="region",
    how="left",
    validate="many_to_one",
)


# ---------------------------------------------------------------------------
# Restrict to regions currently considered storage-accessible
# ---------------------------------------------------------------------------

accessible_crosswalk = crosswalk_gold.loc[
    crosswalk_gold["co2_inject_available"]
].copy()


# ---------------------------------------------------------------------------
# Summarize detailed evidence composition by Gold status
# ---------------------------------------------------------------------------

evidence_breakdown = (
    accessible_crosswalk.groupby(
        [
            "gold_storage_status",
            "source_dataset",
            "assessment_type",
            "data_class",
            "representation",
            "injectivity_status",
        ],
        dropna=False,
    )
    .agg(
        crosswalk_rows=("storage_feature_id", "size"),
        regions=("region", "nunique"),
        storage_units=("storage_unit_id", "nunique"),
        storage_features=("storage_feature_id", "nunique"),
    )
    .reset_index()
    .sort_values(
        [
            "gold_storage_status",
            "regions",
            "crosswalk_rows",
        ],
        ascending=[True, False, False],
    )
)


print("=" * 80)
print("STORAGE EVIDENCE COMPOSITION BY GOLD STATUS")
print("=" * 80)

display(evidence_breakdown)


# ---------------------------------------------------------------------------
# Inspect the ambiguous 'other evidence' class specifically
# ---------------------------------------------------------------------------

other_evidence = accessible_crosswalk.loc[
    accessible_crosswalk["gold_storage_status"]
    == "storage_available_other_evidence"
].copy()


print("\n" + "=" * 80)
print("OTHER STORAGE EVIDENCE — SOURCE DATASETS")
print("=" * 80)

display(
    other_evidence.groupby(
        "source_dataset",
        dropna=False,
    )
    .agg(
        regions=("region", "nunique"),
        storage_units=("storage_unit_id", "nunique"),
        storage_features=("storage_feature_id", "nunique"),
        crosswalk_rows=("storage_feature_id", "size"),
    )
    .reset_index()
    .sort_values("regions", ascending=False)
)


print("\n" + "=" * 80)
print("OTHER STORAGE EVIDENCE — ASSESSMENT TYPES")
print("=" * 80)

display(
    other_evidence.groupby(
        [
            "source_dataset",
            "assessment_type",
            "data_class",
            "representation",
        ],
        dropna=False,
    )
    .agg(
        regions=("region", "nunique"),
        storage_units=("storage_unit_id", "nunique"),
        storage_features=("storage_feature_id", "nunique"),
    )
    .reset_index()
    .sort_values("regions", ascending=False)
)


# ---------------------------------------------------------------------------
# Check whether these regions overlap other evidence categories internally
# ---------------------------------------------------------------------------

region_source_counts = (
    accessible_crosswalk.groupby("region")
    .agg(
        source_count=("source_dataset", "nunique"),
        assessment_type_count=("assessment_type", "nunique"),
        data_class_count=("data_class", "nunique"),
        storage_unit_count=("storage_unit_id", "nunique"),
        storage_feature_count=("storage_feature_id", "nunique"),
    )
    .reset_index()
    .merge(
        gold_storage_regions[
            [
                "region",
                "gold_storage_status",
            ]
        ],
        on="region",
        how="left",
        validate="one_to_one",
    )
)


print("\n" + "=" * 80)
print("OTHER EVIDENCE — REGION COMPLEXITY")
print("=" * 80)

display(
    region_source_counts.loc[
        region_source_counts["gold_storage_status"]
        == "storage_available_other_evidence"
    ]
    .describe()
    .T
)


# ---------------------------------------------------------------------------
# Preview representative rows
# ---------------------------------------------------------------------------

print("\n" + "=" * 80)
print("OTHER EVIDENCE — SAMPLE")
print("=" * 80)

display(
    other_evidence[
        [
            "region",
            "source_dataset",
            "storage_unit_id",
            "storage_type",
            "representation",
            "assessment_type",
            "data_class",
            "capacity_data",
            "injectivity_status",
            "region_overlap_fraction",
            "feature_overlap_fraction",
        ]
    ]
    .drop_duplicates()
    .head(40)
)

STORAGE EVIDENCE COMPOSITION BY GOLD STATUS


,gold_storage_status,source_dataset,assessment_type,data_class,representation,injectivity_status,crosswalk_rows,regions,storage_units,storage_features
2,storage_available_capacity_evidence_unallocated,NATCARB,quantitative_geological_storage_resource,geological_storage_capacity,resource_grid_cell,not_quantitatively_assessed,43554,1481,59,21705
3,storage_available_capacity_evidence_unallocated,NATCARB,quantitative_geological_storage_resource,geological_storage_capacity,storage_resource,not_quantitatively_assessed,2777,750,1341,1341
1,storage_available_capacity_evidence_unallocated,BC_STORAGE_ATLAS,NaN,NaN,pool_extent,NaN,1886,136,1238,1309
0,storage_available_capacity_evidence_unallocated,BC_STORAGE_ATLAS,NaN,NaN,aquifer_extent,NaN,446,136,25,29
4,storage_available_other_evidence,NATCARB,quantitative_geological_storage_resource,geological_storage_capacity,resource_grid_cell,not_quantitatively_assessed,11522,1095,147,5866
5,storage_available_other_evidence,NATCARB,quantitative_geological_storage_resource,geological_storage_capacity,storage_resource,not_quantitatively_assessed,22,16,15,15
6,storage_available_qualitative_only,ATLANTIC_COS,qualitative_chance_of_success,geological_prospectivity,prospectivity_polygon,not_quantitatively_assessed,1541,156,9,712



OTHER STORAGE EVIDENCE — SOURCE DATASETS


,source_dataset,regions,storage_units,storage_features,crosswalk_rows
0,NATCARB,1096,162,5881,11544



OTHER STORAGE EVIDENCE — ASSESSMENT TYPES


,source_dataset,assessment_type,data_class,representation,regions,storage_units,storage_features
0,NATCARB,quantitative_geological_storage_resource,geological_storage_capacity,resource_grid_cell,1095,147,5866
1,NATCARB,quantitative_geological_storage_resource,geological_storage_capacity,storage_resource,16,15,15



OTHER EVIDENCE — REGION COMPLEXITY


,count,mean,std,min,25%,50%,75%,max
source_count,1096.0,1.000000,0.000000,1.0,1.0,1.0,1.0,1.0
assessment_type_count,1096.0,1.000000,0.000000,1.0,1.0,1.0,1.0,1.0
data_class_count,1096.0,1.000000,0.000000,1.0,1.0,1.0,1.0,1.0
storage_unit_count,1096.0,1.413321,1.390695,1.0,1.0,1.0,1.0,20.0
storage_feature_count,1096.0,10.532847,5.555763,1.0,6.0,12.0,14.0,42.0



OTHER EVIDENCE — SAMPLE


,region,source_dataset,storage_unit_id,storage_type,representation,assessment_type,data_class,capacity_data,injectivity_status,region_overlap_fraction,feature_overlap_fraction
759,R1033,NATCARB,NAT_SAL_61c04b2208a1,saline_aquifer,resource_grid_cell,quantitative_geological_storage_resource,geological_storage_capacity,1.0,not_quantitatively_assessed,0.002945,0.018466
760,R1033,NATCARB,NAT_SAL_61c04b2208a1,saline_aquifer,resource_grid_cell,quantitative_geological_storage_resource,geological_storage_capacity,1.0,not_quantitatively_assessed,0.021433,0.134492
761,R1033,NATCARB,NAT_SAL_61c04b2208a1,saline_aquifer,resource_grid_cell,quantitative_geological_storage_resource,geological_storage_capacity,1.0,not_quantitatively_assessed,0.041108,0.258124
762,R1033,NATCARB,NAT_SAL_61c04b2208a1,saline_aquifer,resource_grid_cell,quantitative_geological_storage_resource,geological_storage_capacity,1.0,not_quantitatively_assessed,0.006914,0.043445
763,R1033,NATCARB,NAT_SAL_61c04b2208a1,saline_aquifer,resource_grid_cell,quantitative_geological_storage_resource,geological_storage_capacity,1.0,not_quantitatively_assessed,0.049157,0.308238
764,R1033,NATCARB,NAT_SAL_61c04b2208a1,saline_aquifer,resource_grid_cell,quantitative_geological_storage_resource,geological_storage_capacity,1.0,not_quantitatively_assessed,0.159369,1.000000
765,R1033,NATCARB,NAT_SAL_61c04b2208a1,saline_aquifer,resource_grid_cell,quantitative_geological_storage_resource,geological_storage_capacity,1.0,not_quantitatively_assessed,0.159261,1.000000
766,R1033,NATCARB,NAT_SAL_61c04b2208a1,saline_aquifer,resource_grid_cell,quantitative_geological_storage_resource,geological_storage_capacity,1.0,not_quantitatively_assessed,0.034048,0.213931
767,R1033,NATCARB,NAT_SAL_61c04b2208a1,saline_aquifer,resource_grid_cell,quantitative_geological_storage_resource,geological_storage_capacity,1.0,not_quantitatively_assessed,0.029734,0.186444
768,R1033,NATCARB,NAT_SAL_61c04b2208a1,saline_aquifer,resource_grid_cell,quantitative_geological_storage_resource,geological_storage_capacity,1.0,not_quantitatively_assessed,0.159371,1.000000


In [9]:
# ---------------------------------------------------------------------------
# Cell 11 — Refine Gold storage evidence classification
# ---------------------------------------------------------------------------

def classify_storage_status(row):
    """Classify regional storage evidence by its current Gold usefulness."""

    if not row["storage_accessible"]:
        return "no_storage_evidence"

    if row["has_any_capacity_evidence"]:
        return "storage_with_positive_capacity_assessment"

    if row["has_quantitative_storage_evidence"]:
        return "quantitative_storage_evidence_no_positive_capacity"

    if row["has_qualitative_storage_evidence"]:
        return "qualitative_storage_evidence"

    return "unclassified_storage_evidence"


gold_storage_regions["gold_storage_status"] = (
    gold_storage_regions.apply(
        classify_storage_status,
        axis=1,
    )
)


# ---------------------------------------------------------------------------
# Rebuild candidate injection-region table
# ---------------------------------------------------------------------------

co2_inject_regions = (
    gold_storage_regions.loc[
        gold_storage_regions["co2_inject_available"]
    ]
    .copy()
    .sort_values("region")
    .reset_index(drop=True)
)


# ---------------------------------------------------------------------------
# Summarize revised evidence classes
# ---------------------------------------------------------------------------

status_summary = (
    gold_storage_regions["gold_storage_status"]
    .value_counts()
    .rename_axis("gold_storage_status")
    .reset_index(name="regions")
)

status_summary["share_of_all_regions_percent"] = (
    status_summary["regions"]
    / len(gold_storage_regions)
    * 100
)

display(status_summary)


print("\n" + "=" * 80)
print("CO2_INJECT-ELIGIBLE REGIONS")
print("=" * 80)

inject_summary = (
    co2_inject_regions["gold_storage_status"]
    .value_counts()
    .rename_axis("gold_storage_status")
    .reset_index(name="regions")
)

inject_summary["share_of_inject_regions_percent"] = (
    inject_summary["regions"]
    / len(co2_inject_regions)
    * 100
)

display(inject_summary)

,gold_storage_status,regions,share_of_all_regions_percent
0,no_storage_evidence,6534,70.493041
1,storage_with_positive_capacity_assessment,1483,15.999568
2,quantitative_storage_evidence_no_positive_capa...,1096,11.824361
3,qualitative_storage_evidence,156,1.683029



CO2_INJECT-ELIGIBLE REGIONS


,gold_storage_status,regions,share_of_inject_regions_percent
0,storage_with_positive_capacity_assessment,1483,54.223035
1,quantitative_storage_evidence_no_positive_capa...,1096,40.073126
2,qualitative_storage_evidence,156,5.703839


no_storage_evidence
    -> CO2_INJECT unavailable

storage_with_positive_capacity_assessment
    -> CO2_INJECT available

quantitative_storage_evidence_no_positive_capacity
    -> CO2_INJECT available

qualitative_storage_evidence
    -> CO2_INJECT available

In [10]:
# ---------------------------------------------------------------------------
# Cell 12 — Build candidate Gold CO2_INJECT availability rows
# ---------------------------------------------------------------------------


# ---------------------------------------------------------------------------
# Candidate regions where CO2_INJECT may exist
# ---------------------------------------------------------------------------

inject_regions = (
    gold_storage_regions.loc[
        gold_storage_regions["co2_inject_available"],
        [
            "region",
            "gold_storage_status",
            "has_quantitative_storage_evidence",
            "has_qualitative_storage_evidence",
            "has_any_capacity_evidence",
            "capacity_evidence_coverage_fraction",
        ],
    ]
    .copy()
    .sort_values("region")
    .reset_index(drop=True)
)


# ---------------------------------------------------------------------------
# Candidate Efficiency rows
#
# Current working representation:
#
#     co2 -> CO2_INJECT -> co2_stored
#
# One process row is required for every region where injection is allowed.
# ---------------------------------------------------------------------------

candidate_injection_efficiency = pd.DataFrame(
    {
        "region": inject_regions["region"],
        "tech": "CO2_INJECT",
        "vintage": 1,
        "input_comm": "co2",
        "output_comm": "co2_stored",
        "efficiency": 1.0,
        "notes": (
            "Geological CO2 injection enabled from mapped Silver storage evidence"
        ),
        "data_source": "CanCO2 unified storage Silver layer",
        "dq_cred": None,
        "dq_geog": None,
        "dq_struc": None,
        "dq_tech": None,
        "dq_time": None,
        "data_id": "GEO_STORAGE",
    }
)


# ---------------------------------------------------------------------------
# Candidate constraint subsets
#
# These identify where constraints may eventually be populated,
# but do NOT yet create numerical LimitResource or LimitActivity rows.
# ---------------------------------------------------------------------------

candidate_resource_regions = (
    inject_regions.loc[
        inject_regions["has_any_capacity_evidence"],
        [
            "region",
            "gold_storage_status",
            "capacity_evidence_coverage_fraction",
        ],
    ]
    .copy()
    .reset_index(drop=True)
)

candidate_activity_regions = inject_regions.iloc[0:0].copy()


# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------

print("=" * 80)
print("CANDIDATE CO2_INJECT GOLD AVAILABILITY")
print("=" * 80)

print(f"Injection-enabled regions: {len(inject_regions):,}")
print(f"Candidate Efficiency rows: {len(candidate_injection_efficiency):,}")

display(candidate_injection_efficiency.head(20))


print("\n" + "=" * 80)
print("CANDIDATE FUTURE LIMITRESOURCE REGIONS")
print("=" * 80)

print(
    f"Regions with positive capacity assessment evidence: "
    f"{len(candidate_resource_regions):,}"
)

display(candidate_resource_regions.head(20))


print("\n" + "=" * 80)
print("CANDIDATE FUTURE LIMITACTIVITY REGIONS")
print("=" * 80)

print(
    "No quantitative injectivity limits are currently ready for encoding."
)

CANDIDATE CO2_INJECT GOLD AVAILABILITY
Injection-enabled regions: 2,735
Candidate Efficiency rows: 2,735


,region,tech,vintage,input_comm,output_comm,efficiency,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
0,R0,CO2_INJECT,1,co2,co2_stored,1.0,Geological CO2 injection enabled from mapped S...,CanCO2 unified storage Silver layer,None,None,None,None,None,GEO_STORAGE
1,R1001,CO2_INJECT,1,co2,co2_stored,1.0,Geological CO2 injection enabled from mapped S...,CanCO2 unified storage Silver layer,None,None,None,None,None,GEO_STORAGE
2,R1002,CO2_INJECT,1,co2,co2_stored,1.0,Geological CO2 injection enabled from mapped S...,CanCO2 unified storage Silver layer,None,None,None,None,None,GEO_STORAGE
3,R1003,CO2_INJECT,1,co2,co2_stored,1.0,Geological CO2 injection enabled from mapped S...,CanCO2 unified storage Silver layer,None,None,None,None,None,GEO_STORAGE
4,R1004,CO2_INJECT,1,co2,co2_stored,1.0,Geological CO2 injection enabled from mapped S...,CanCO2 unified storage Silver layer,None,None,None,None,None,GEO_STORAGE
5,R1005,CO2_INJECT,1,co2,co2_stored,1.0,Geological CO2 injection enabled from mapped S...,CanCO2 unified storage Silver layer,None,None,None,None,None,GEO_STORAGE
6,R1006,CO2_INJECT,1,co2,co2_stored,1.0,Geological CO2 injection enabled from mapped S...,CanCO2 unified storage Silver layer,None,None,None,None,None,GEO_STORAGE
7,R1007,CO2_INJECT,1,co2,co2_stored,1.0,Geological CO2 injection enabled from mapped S...,CanCO2 unified storage Silver layer,None,None,None,None,None,GEO_STORAGE
8,R1008,CO2_INJECT,1,co2,co2_stored,1.0,Geological CO2 injection enabled from mapped S...,CanCO2 unified storage Silver layer,None,None,None,None,None,GEO_STORAGE
9,R1009,CO2_INJECT,1,co2,co2_stored,1.0,Geological CO2 injection enabled from mapped S...,CanCO2 unified storage Silver layer,None,None,None,None,None,GEO_STORAGE



CANDIDATE FUTURE LIMITRESOURCE REGIONS
Regions with positive capacity assessment evidence: 1,483


,region,gold_storage_status,capacity_evidence_coverage_fraction
0,R0,storage_with_positive_capacity_assessment,0.005919
1,R1016,storage_with_positive_capacity_assessment,0.696139
2,R1017,storage_with_positive_capacity_assessment,0.836223
3,R1018,storage_with_positive_capacity_assessment,0.990102
4,R1019,storage_with_positive_capacity_assessment,1.000000
5,R1020,storage_with_positive_capacity_assessment,1.000000
6,R1021,storage_with_positive_capacity_assessment,1.000000
7,R1022,storage_with_positive_capacity_assessment,1.000000
8,R1023,storage_with_positive_capacity_assessment,1.000000
9,R1024,storage_with_positive_capacity_assessment,1.000000



CANDIDATE FUTURE LIMITACTIVITY REGIONS
No quantitative injectivity limits are currently ready for encoding.


In [11]:
# ---------------------------------------------------------------------------
# Cell 13 — Validate candidate CO2_INJECT rows against Efficiency schema
# ---------------------------------------------------------------------------


with sqlite3.connect(db_path) as conn:
    efficiency_schema = pd.read_sql_query(
        "PRAGMA table_info(Efficiency)",
        conn,
    )

    existing_efficiency = pd.read_sql_query(
        "SELECT * FROM Efficiency LIMIT 10",
        conn,
    )


print("=" * 80)
print("EFFICIENCY TABLE SCHEMA")
print("=" * 80)

display(efficiency_schema)


print("\n" + "=" * 80)
print("CURRENT EFFICIENCY ROWS")
print("=" * 80)

display(existing_efficiency)


# ---------------------------------------------------------------------------
# Compare candidate columns with actual schema
# ---------------------------------------------------------------------------

actual_efficiency_columns = efficiency_schema["name"].tolist()
candidate_columns = candidate_injection_efficiency.columns.tolist()

print("\n" + "=" * 80)
print("SCHEMA COMPARISON")
print("=" * 80)

print("Actual Efficiency columns:")
print(actual_efficiency_columns)

print("\nCandidate columns:")
print(candidate_columns)

print(
    "\nMissing from candidate:",
    sorted(set(actual_efficiency_columns) - set(candidate_columns)),
)

print(
    "Extra in candidate:",
    sorted(set(candidate_columns) - set(actual_efficiency_columns)),
)


# ---------------------------------------------------------------------------
# Reorder candidate rows where schemas already match
# ---------------------------------------------------------------------------

if set(actual_efficiency_columns) == set(candidate_columns):

    candidate_injection_efficiency = (
        candidate_injection_efficiency[
            actual_efficiency_columns
        ]
        .copy()
    )

    print("\nCandidate CO2_INJECT rows exactly match Efficiency schema.")

else:
    print(
        "\nCandidate rows require adjustment before they should be "
        "written into the Gold schema."
    )

EFFICIENCY TABLE SCHEMA


,cid,name,type,notnull,dflt_value,pk
0,0,region,TEXT,0,None,1
1,1,input_comm,TEXT,0,None,2
2,2,tech,TEXT,0,None,3
3,3,vintage,INTEGER,0,None,4
4,4,output_comm,TEXT,0,None,5
5,5,efficiency,REAL,0,None,0
6,6,notes,TEXT,0,None,0
7,7,data_source,TEXT,0,None,0
8,8,dq_cred,INTEGER,0,None,0
9,9,dq_geog,INTEGER,0,None,0



CURRENT EFFICIENCY ROWS


,region,input_comm,tech,vintage,output_comm,efficiency,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id



SCHEMA COMPARISON
Actual Efficiency columns:
['region', 'input_comm', 'tech', 'vintage', 'output_comm', 'efficiency', 'notes', 'data_source', 'dq_cred', 'dq_geog', 'dq_struc', 'dq_tech', 'dq_time', 'data_id']

Candidate columns:
['region', 'tech', 'vintage', 'input_comm', 'output_comm', 'efficiency', 'notes', 'data_source', 'dq_cred', 'dq_geog', 'dq_struc', 'dq_tech', 'dq_time', 'data_id']

Missing from candidate: []
Extra in candidate: []

Candidate CO2_INJECT rows exactly match Efficiency schema.


In [12]:
# ---------------------------------------------------------------------------
# Cell 14 — Validate CO2_INJECT and co2_stored against model schemas
# ---------------------------------------------------------------------------


with sqlite3.connect(db_path) as conn:
    technology_schema = pd.read_sql_query(
        "PRAGMA table_info(Technology)",
        conn,
    )

    commodity_schema = pd.read_sql_query(
        "PRAGMA table_info(Commodity)",
        conn,
    )

    existing_technologies = pd.read_sql_query(
        "SELECT * FROM Technology LIMIT 10",
        conn,
    )

    existing_commodities = pd.read_sql_query(
        "SELECT * FROM Commodity LIMIT 10",
        conn,
    )


# ---------------------------------------------------------------------------
# Inspect schemas
# ---------------------------------------------------------------------------

print("=" * 80)
print("TECHNOLOGY TABLE SCHEMA")
print("=" * 80)

display(technology_schema)

print("\nCurrent Technology rows:")
display(existing_technologies)


print("\n" + "=" * 80)
print("COMMODITY TABLE SCHEMA")
print("=" * 80)

display(commodity_schema)

print("\nCurrent Commodity rows:")
display(existing_commodities)


# ---------------------------------------------------------------------------
# Candidate registry additions from earlier design
# ---------------------------------------------------------------------------

candidate_technology = pd.DataFrame(
    [
        {
            "tech": "CO2_INJECT",
            "flag": "p",
            "annual": 1,
            "exchange": 0,
            "unlim_cap": 0,
        }
    ]
)

candidate_commodity = pd.DataFrame(
    [
        {
            "name": "co2_stored",
            "flag": "wa",
            "description": "geologically stored co2",
        }
    ]
)


# ---------------------------------------------------------------------------
# Compare against actual database schemas
# ---------------------------------------------------------------------------

technology_columns = technology_schema["name"].tolist()
commodity_columns = commodity_schema["name"].tolist()

print("\n" + "=" * 80)
print("TECHNOLOGY SCHEMA COMPARISON")
print("=" * 80)

print("Actual columns:")
print(technology_columns)

print("\nCandidate columns:")
print(candidate_technology.columns.tolist())

print(
    "\nMissing from candidate:",
    sorted(set(technology_columns) - set(candidate_technology.columns)),
)

print(
    "Extra in candidate:",
    sorted(set(candidate_technology.columns) - set(technology_columns)),
)


print("\n" + "=" * 80)
print("COMMODITY SCHEMA COMPARISON")
print("=" * 80)

print("Actual columns:")
print(commodity_columns)

print("\nCandidate columns:")
print(candidate_commodity.columns.tolist())

print(
    "\nMissing from candidate:",
    sorted(set(commodity_columns) - set(candidate_commodity.columns)),
)

print(
    "Extra in candidate:",
    sorted(set(candidate_commodity.columns) - set(commodity_columns)),
)

TECHNOLOGY TABLE SCHEMA


,cid,name,type,notnull,dflt_value,pk
0,0,tech,TEXT,1,None,1
1,1,flag,TEXT,1,None,0
2,2,sector,TEXT,0,None,0
3,3,category,TEXT,0,None,0
4,4,sub_category,TEXT,0,None,0
5,5,unlim_cap,INTEGER,1,0,0
6,6,annual,INTEGER,1,0,0
7,7,reserve,INTEGER,1,0,0
8,8,curtail,INTEGER,1,0,0
9,9,retire,INTEGER,1,0,0



Current Technology rows:


,tech,flag,sector,category,sub_category,unlim_cap,annual,reserve,curtail,retire,flex,exchange,seas_stor,description,data_id



COMMODITY TABLE SCHEMA


,cid,name,type,notnull,dflt_value,pk
0,0,name,TEXT,0,None,1
1,1,flag,TEXT,0,None,0
2,2,description,TEXT,0,None,0
3,3,data_id,TEXT,0,None,2



Current Commodity rows:


,name,flag,description,data_id



TECHNOLOGY SCHEMA COMPARISON
Actual columns:
['tech', 'flag', 'sector', 'category', 'sub_category', 'unlim_cap', 'annual', 'reserve', 'curtail', 'retire', 'flex', 'exchange', 'seas_stor', 'description', 'data_id']

Candidate columns:
['tech', 'flag', 'annual', 'exchange', 'unlim_cap']

Missing from candidate: ['category', 'curtail', 'data_id', 'description', 'flex', 'reserve', 'retire', 'seas_stor', 'sector', 'sub_category']
Extra in candidate: []

COMMODITY SCHEMA COMPARISON
Actual columns:
['name', 'flag', 'description', 'data_id']

Candidate columns:
['name', 'flag', 'description']

Missing from candidate: ['data_id']
Extra in candidate: []


In [13]:
# ---------------------------------------------------------------------------
# Cell 15 — Expand storage registry definitions to Gold-schema rows
# ---------------------------------------------------------------------------


# ---------------------------------------------------------------------------
# Technology row
# ---------------------------------------------------------------------------

candidate_technology_gold = pd.DataFrame(
    [
        {
            "tech": "CO2_INJECT",
            "flag": "p",
            "sector": None,
            "category": "co2_storage",
            "sub_category": "geological_injection",
            "unlim_cap": 0,
            "annual": 1,
            "reserve": 0,
            "curtail": 0,
            "retire": 0,
            "flex": 0,
            "exchange": 0,
            "seas_stor": 0,
            "description": "Geological CO2 injection and permanent storage",
            "data_id": "GEO_STORAGE",
        }
    ]
)


# ---------------------------------------------------------------------------
# Commodity row
# ---------------------------------------------------------------------------

candidate_commodity_gold = pd.DataFrame(
    [
        {
            "name": "co2_stored",
            "flag": "wa",
            "description": "Geologically stored CO2",
            "data_id": "GEO_STORAGE",
        }
    ]
)


# ---------------------------------------------------------------------------
# Reorder explicitly to match SQLite schema
# ---------------------------------------------------------------------------

candidate_technology_gold = candidate_technology_gold[
    technology_columns
]

candidate_commodity_gold = candidate_commodity_gold[
    commodity_columns
]


# ---------------------------------------------------------------------------
# Validate exact schema match
# ---------------------------------------------------------------------------

assert candidate_technology_gold.columns.tolist() == technology_columns
assert candidate_commodity_gold.columns.tolist() == commodity_columns


print("=" * 80)
print("CO2_INJECT TECHNOLOGY ROW")
print("=" * 80)

display(candidate_technology_gold)


print("\n" + "=" * 80)
print("CO2_STORED COMMODITY ROW")
print("=" * 80)

display(candidate_commodity_gold)

CO2_INJECT TECHNOLOGY ROW


,tech,flag,sector,category,sub_category,unlim_cap,annual,reserve,curtail,retire,flex,exchange,seas_stor,description,data_id
0,CO2_INJECT,p,None,co2_storage,geological_injection,0,1,0,0,0,0,0,0,Geological CO2 injection and permanent storage,GEO_STORAGE



CO2_STORED COMMODITY ROW


,name,flag,description,data_id
0,co2_stored,wa,Geologically stored CO2,GEO_STORAGE


In [14]:
# ---------------------------------------------------------------------------
# Cell 16 — Prototype storage-specific Efficiency integration
# ---------------------------------------------------------------------------


# ---------------------------------------------------------------------------
# Existing node-efficiency logic cannot directly handle CO2_INJECT
#
# Most technologies in generation_efficiency.csv are expanded across all
# graph-node regions. CO2_INJECT is spatially restricted by the storage Silver
# layer, so it requires a separate regional expansion step.
# ---------------------------------------------------------------------------

storage_efficiency_rows = pd.DataFrame(
    {
        "region": co2_inject_regions["region"],
        "input_comm": "co2",
        "tech": "CO2_INJECT",
        "vintage": 1,
        "output_comm": "co2_stored",
        "efficiency": 1.0,
        "notes": (
            "CO2 injection availability derived from mapped geological "
            "storage evidence"
        ),
        "data_source": "CanCO2 unified storage Silver layer",
        "dq_cred": None,
        "dq_geog": None,
        "dq_struc": None,
        "dq_tech": None,
        "dq_time": None,
        "data_id": "GEO_STORAGE",
    }
)


# ---------------------------------------------------------------------------
# Match actual Efficiency schema
# ---------------------------------------------------------------------------

storage_efficiency_rows = storage_efficiency_rows[
    actual_efficiency_columns
].copy()


# ---------------------------------------------------------------------------
# Structural validation
# ---------------------------------------------------------------------------

assert len(storage_efficiency_rows) == len(co2_inject_regions)

assert storage_efficiency_rows["region"].nunique() == len(
    storage_efficiency_rows
)

assert set(storage_efficiency_rows["region"]) == set(
    co2_inject_regions["region"]
)

assert storage_efficiency_rows["tech"].eq("CO2_INJECT").all()
assert storage_efficiency_rows["input_comm"].eq("co2").all()
assert storage_efficiency_rows["output_comm"].eq("co2_stored").all()
assert storage_efficiency_rows["efficiency"].eq(1.0).all()


print("=" * 80)
print("STORAGE-SPECIFIC EFFICIENCY INTEGRATION")
print("=" * 80)

print(
    f"CO2_INJECT Efficiency rows: "
    f"{len(storage_efficiency_rows):,}"
)

print(
    f"Unique injection-enabled regions: "
    f"{storage_efficiency_rows['region'].nunique():,}"
)

print(
    f"Excluded model regions: "
    f"{len(gold_storage_regions) - len(storage_efficiency_rows):,}"
)

display(storage_efficiency_rows.head(20))

STORAGE-SPECIFIC EFFICIENCY INTEGRATION
CO2_INJECT Efficiency rows: 2,735
Unique injection-enabled regions: 2,735
Excluded model regions: 6,534


,region,input_comm,tech,vintage,output_comm,efficiency,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
0,R0,co2,CO2_INJECT,1,co2_stored,1.0,CO2 injection availability derived from mapped...,CanCO2 unified storage Silver layer,None,None,None,None,None,GEO_STORAGE
1,R1001,co2,CO2_INJECT,1,co2_stored,1.0,CO2 injection availability derived from mapped...,CanCO2 unified storage Silver layer,None,None,None,None,None,GEO_STORAGE
2,R1002,co2,CO2_INJECT,1,co2_stored,1.0,CO2 injection availability derived from mapped...,CanCO2 unified storage Silver layer,None,None,None,None,None,GEO_STORAGE
3,R1003,co2,CO2_INJECT,1,co2_stored,1.0,CO2 injection availability derived from mapped...,CanCO2 unified storage Silver layer,None,None,None,None,None,GEO_STORAGE
4,R1004,co2,CO2_INJECT,1,co2_stored,1.0,CO2 injection availability derived from mapped...,CanCO2 unified storage Silver layer,None,None,None,None,None,GEO_STORAGE
5,R1005,co2,CO2_INJECT,1,co2_stored,1.0,CO2 injection availability derived from mapped...,CanCO2 unified storage Silver layer,None,None,None,None,None,GEO_STORAGE
6,R1006,co2,CO2_INJECT,1,co2_stored,1.0,CO2 injection availability derived from mapped...,CanCO2 unified storage Silver layer,None,None,None,None,None,GEO_STORAGE
7,R1007,co2,CO2_INJECT,1,co2_stored,1.0,CO2 injection availability derived from mapped...,CanCO2 unified storage Silver layer,None,None,None,None,None,GEO_STORAGE
8,R1008,co2,CO2_INJECT,1,co2_stored,1.0,CO2 injection availability derived from mapped...,CanCO2 unified storage Silver layer,None,None,None,None,None,GEO_STORAGE
9,R1009,co2,CO2_INJECT,1,co2_stored,1.0,CO2 injection availability derived from mapped...,CanCO2 unified storage Silver layer,None,None,None,None,None,GEO_STORAGE


## Storage-specific process expansion

The existing GeoCANOE schema builder expands most node technologies from
`generation_efficiency.csv` across every graph-node region.

This is appropriate for technologies whose regional availability is not
controlled by an external spatial resource layer.

`CO2_INJECT` is different.

Its process definition is globally described by:

```text
co2 -> CO2_INJECT -> co2_stored

generation_efficiency.csv
        |
        +--> ordinary node technologies
        |        expanded using existing node rules
        |
        +--> CO2_INJECT definition
                 |
                 v
        intersect with storage-enabled regions
                 |
                 v
        regional CO2_INJECT Efficiency rows

In [15]:
# ---------------------------------------------------------------------------
# Cell 17 — Prototype rebuild_storage_efficiency()
# ---------------------------------------------------------------------------


def rebuild_storage_efficiency(
    db_encoded: dict[str, pd.DataFrame],
    storage_regions: pd.DataFrame,
) -> None:
    """Append region-specific geological CO2 injection process rows.

    Geological storage availability is derived from the Silver storage layer.
    Only regions with ``storage_accessible == True`` receive the ``CO2_INJECT``
    process.

    The physical process is:

        co2 -> CO2_INJECT -> co2_stored

    Parameters
    ----------
    db_encoded : dict[str, pd.DataFrame]
        Mutable mapping of CANOE/TEMOA table names to encoded DataFrames.

    storage_regions : pd.DataFrame
        Regional Silver storage-evidence table containing at minimum
        ``region`` and ``storage_accessible``.

    Returns
    -------
    None
        ``db_encoded["Efficiency"]`` is modified in place.
    """

    required_columns = {
        "region",
        "storage_accessible",
    }

    missing_columns = required_columns - set(storage_regions.columns)

    if missing_columns:
        raise ValueError(
            "Storage regional evidence is missing required columns: "
            f"{sorted(missing_columns)}"
        )

    eligible_regions = (
        storage_regions.loc[
            storage_regions["storage_accessible"],
            "region",
        ]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    storage_efficiency = pd.DataFrame(
        {
            "region": eligible_regions,
            "input_comm": "co2",
            "tech": "CO2_INJECT",
            "vintage": 1,
            "output_comm": "co2_stored",
            "efficiency": 1.0,
            "notes": (
                "CO2 injection availability derived from mapped "
                "geological storage evidence"
            ),
            "data_source": "CanCO2 unified storage Silver layer",
            "dq_cred": None,
            "dq_geog": None,
            "dq_struc": None,
            "dq_tech": None,
            "dq_time": None,
            "data_id": "GEO_STORAGE",
        }
    )

    # Preserve the existing Efficiency schema exactly.
    storage_efficiency = storage_efficiency[
        db_encoded["Efficiency"].columns
    ].copy()

    # -----------------------------------------------------------------------
    # Remove stale CO2_INJECT rows before rebuilding.
    # -----------------------------------------------------------------------

    existing_efficiency = db_encoded["Efficiency"].loc[
        db_encoded["Efficiency"]["tech"] != "CO2_INJECT"
    ].copy()

    # Avoid concatenating against an empty/all-NA frame. This also prevents
    # the Pandas FutureWarning associated with changing dtype inference.
    if existing_efficiency.empty:
        db_encoded["Efficiency"] = storage_efficiency.copy()
    else:
        db_encoded["Efficiency"] = pd.concat(
            [
                existing_efficiency,
                storage_efficiency,
            ],
            ignore_index=True,
        )

    # -----------------------------------------------------------------------
    # Validation
    # -----------------------------------------------------------------------

    encoded_storage = db_encoded["Efficiency"].loc[
        db_encoded["Efficiency"]["tech"] == "CO2_INJECT"
    ].copy()

    assert len(encoded_storage) == len(eligible_regions)

    assert encoded_storage["region"].nunique() == len(
        eligible_regions
    )

    assert set(encoded_storage["region"]) == set(
        eligible_regions
    )

    assert encoded_storage["input_comm"].eq("co2").all()
    assert encoded_storage["output_comm"].eq("co2_stored").all()
    assert encoded_storage["efficiency"].eq(1.0).all()

    print(
        f"CO2_INJECT Efficiency rows added: "
        f"{len(storage_efficiency):,}"
    )

In [16]:
# ---------------------------------------------------------------------------
# Cell 18 — Test rebuild_storage_efficiency() on a scratch table
# ---------------------------------------------------------------------------


# ---------------------------------------------------------------------------
# Load the current Efficiency table only to preserve its exact schema
# ---------------------------------------------------------------------------

with sqlite3.connect(db_path) as conn:
    efficiency_base = pd.read_sql_query(
        "SELECT * FROM Efficiency",
        conn,
    )


db_storage_test = {
    "Efficiency": efficiency_base.copy(),
}


# ---------------------------------------------------------------------------
# Run prototype storage rebuild
# ---------------------------------------------------------------------------

rebuild_storage_efficiency(
    db_encoded=db_storage_test,
    storage_regions=storage_regions,
)


# ---------------------------------------------------------------------------
# Inspect result
# ---------------------------------------------------------------------------

storage_test_rows = db_storage_test["Efficiency"].loc[
    db_storage_test["Efficiency"]["tech"] == "CO2_INJECT"
].copy()


print("=" * 80)
print("STORAGE EFFICIENCY PROTOTYPE TEST")
print("=" * 80)

print(f"Rows: {len(storage_test_rows):,}")
print(
    f"Unique regions: "
    f"{storage_test_rows['region'].nunique():,}"
)

print(
    f"Expected eligible regions: "
    f"{storage_regions['storage_accessible'].sum():,}"
)

display(storage_test_rows.head(20))

CO2_INJECT Efficiency rows added: 2,735
STORAGE EFFICIENCY PROTOTYPE TEST
Rows: 2,735
Unique regions: 2,735
Expected eligible regions: 2,735


,region,input_comm,tech,vintage,output_comm,efficiency,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
0,R0,co2,CO2_INJECT,1,co2_stored,1.0,CO2 injection availability derived from mapped...,CanCO2 unified storage Silver layer,None,None,None,None,None,GEO_STORAGE
1,R4,co2,CO2_INJECT,1,co2_stored,1.0,CO2 injection availability derived from mapped...,CanCO2 unified storage Silver layer,None,None,None,None,None,GEO_STORAGE
2,R33,co2,CO2_INJECT,1,co2_stored,1.0,CO2 injection availability derived from mapped...,CanCO2 unified storage Silver layer,None,None,None,None,None,GEO_STORAGE
3,R565,co2,CO2_INJECT,1,co2_stored,1.0,CO2 injection availability derived from mapped...,CanCO2 unified storage Silver layer,None,None,None,None,None,GEO_STORAGE
4,R636,co2,CO2_INJECT,1,co2_stored,1.0,CO2 injection availability derived from mapped...,CanCO2 unified storage Silver layer,None,None,None,None,None,GEO_STORAGE
5,R637,co2,CO2_INJECT,1,co2_stored,1.0,CO2 injection availability derived from mapped...,CanCO2 unified storage Silver layer,None,None,None,None,None,GEO_STORAGE
6,R638,co2,CO2_INJECT,1,co2_stored,1.0,CO2 injection availability derived from mapped...,CanCO2 unified storage Silver layer,None,None,None,None,None,GEO_STORAGE
7,R712,co2,CO2_INJECT,1,co2_stored,1.0,CO2 injection availability derived from mapped...,CanCO2 unified storage Silver layer,None,None,None,None,None,GEO_STORAGE
8,R713,co2,CO2_INJECT,1,co2_stored,1.0,CO2 injection availability derived from mapped...,CanCO2 unified storage Silver layer,None,None,None,None,None,GEO_STORAGE
9,R715,co2,CO2_INJECT,1,co2_stored,1.0,CO2 injection availability derived from mapped...,CanCO2 unified storage Silver layer,None,None,None,None,None,GEO_STORAGE


# 18 — Geological CO₂ Storage Representation in GeoCANOE / Temoa

## Conclusion

This notebook established the initial modelling architecture for geological CO₂ storage within the GeoCANOE / Temoa workflow.

The main objective was not to complete a full geological storage cost-and-capacity representation, but to determine how storage should enter the model structurally and how the existing Silver geological-storage products should inform Gold-schema construction.

The working physical representation is:

```text
co2
 |
 v
CO2_INJECT
 |
 v
co2_stored